# ELENDRIL V8 — Typed Action / Fact-Pointer Screen (Kaggle 2xT4)

Execution notebook pinned to immutable runtime commit:
- `SOURCE_SHA = "dd12e7e2ed040a84a0ba27ec566b1715ecc59b4f"`
- Defaults are inert (`RUN_STUDY = False`) until explicitly enabled by the owner.


In [ ]:
# Owner-controlled switches (keep RUN_STUDY=False until ready to run on Kaggle 2xT4)
RUN_STUDY = False
PHASE = "data-control"  # archive-diagnostic | data-control | head-screen | reader-screen | replication
MINUTES = 330
WINNER_ARM = None
COMPARATOR_ARM = None
CONFIRMATION_SEED = None
SOURCE_SHA = "dd12e7e2ed040a84a0ba27ec566b1715ecc59b4f"
REPO_URL = "https://github.com/dhurv0045com-spec/An-Ra-the-new-AGI.git"


In [ ]:
import os, shutil, subprocess, tarfile, urllib.request
from pathlib import Path

WORK_ROOT = Path("/kaggle/working/elendril_v8")
REPO_DIR = WORK_ROOT / "repo"
OUTPUT_DIR = Path("/kaggle/working")
WORK_ROOT.mkdir(parents=True, exist_ok=True)

if not RUN_STUDY:
    print("NOT LAUNCHED: owner switch is false")
else:
    if REPO_DIR.exists():
        shutil.rmtree(REPO_DIR)
    REPO_DIR.mkdir(parents=True, exist_ok=True)
    local_tar = Path(f"/kaggle/input/elendril-v8-source/{SOURCE_SHA}.tar.gz")
    tar_path = WORK_ROOT / f"{SOURCE_SHA}.tar.gz"
    if local_tar.exists():
        shutil.copy2(local_tar, tar_path)
    else:
        subprocess.run(["git", "clone", "--no-checkout", REPO_URL, str(REPO_DIR)], check=True)
        subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "origin", SOURCE_SHA], check=True)
        subprocess.run(["git", "-C", str(REPO_DIR), "checkout", "--detach", SOURCE_SHA], check=True)
    if tar_path.exists():
        with tarfile.open(tar_path, "r:gz") as tf:
            tf.extractall(REPO_DIR)
    os.chdir(REPO_DIR)
    print("Checked out immutable runtime:", SOURCE_SHA)


In [ ]:
import torch
if RUN_STUDY:
    print("Torch:", torch.__version__, "CUDA GPUs:", torch.cuda.device_count())
    for idx in range(torch.cuda.device_count()):
        print(f"  cuda:{idx} -> {torch.cuda.get_device_name(idx)}")
    assert torch.cuda.is_available(), "Kaggle GPU accelerator must be enabled"


In [ ]:
import sys, subprocess
if not RUN_STUDY:
    print("NOT LAUNCHED: owner switch is false")
else:
    cmd = [
        sys.executable, "-m", "tools.elendril_v8_campaign",
        "--phase", PHASE,
        "--source", SOURCE_SHA,
        "--work-root", str(WORK_ROOT / "runs"),
        "--output", str(OUTPUT_DIR),
        "--minutes", str(MINUTES),
        "--owner-enabled",
    ]
    if WINNER_ARM:
        cmd.extend(["--winner-arm", str(WINNER_ARM)])
    if COMPARATOR_ARM:
        cmd.extend(["--comparator-arm", str(COMPARATOR_ARM)])
    if CONFIRMATION_SEED is not None:
        cmd.extend(["--confirmation-seed", str(CONFIRMATION_SEED)])
    subprocess.run(cmd, check=True)


In [ ]:
import json
from IPython.display import FileLink, display
from tools.elendril_v8_export import verify_v8_export

if not RUN_STUDY:
    print("NOT LAUNCHED: owner switch is false")
else:
    receipt = verify_v8_export(OUTPUT_DIR, expected_source_commit=SOURCE_SHA)
    print("Verified exports:", json.dumps(receipt, indent=2))
    for fname in ("FINAL_STATUS.json", "ELENDRIL_V8_ANALYSIS.zip", "ELENDRIL_V8_FULL.zip"):
        if (OUTPUT_DIR / fname).exists():
            display(FileLink(str(OUTPUT_DIR / fname)))
